In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    get_stimulus_onset_samples,
    DEFAULT_STIMULUS_LABEL,
)
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
    PreprocessedDataVariants,
    SingleDataMetadata,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# Stimulus Alignment — ASSR

Aligns stimulus onsets (annotated `fam+`) across the participants of one group so
that stimulus *k* falls at the **same sample index** in every recording, enabling
sample-locked cross-participant analysis.

The workflow is two-stage (see `src/preprocessing/stimulus_alignment.py`):

1. **Coarse crop** (during preprocessing): each recording is trimmed by up to
   `trim_sec` from each end, never closer than `min_keep_sec` to the first/last
   onset. Continuous (no splicing) so filtering/ICA are clean.
2. **Fine alignment** (here): every inter-stimulus interval is trimmed from the
   front to the group-wide minimum (preserving `keep_tail` of data before each
   onset), and the EEG is spliced. Edges keep the shortest remaining lead-in/out.

This notebook runs the fine alignment on the preprocessed (`after_ica`) data and
inspects the result.

> **Requires preprocessing to have run** for the selected experiment
> (`python scripts/run_preprocessing.py --experiment assr --raw_processing`).

## Configuration

In [ ]:
# ── Group to align ────────────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR  # ASSR has a single (placeholder) music type
EXCLUSION_CATEGORIES: list[ExclusionCategories] = [ExclusionCategories.WRONG_CONDITION]  # e.g. [ExclusionCategories.ARTIFACTS]

# ── Alignment parameters ──────────────────────────────────────────────────────
STIMULUS_LABEL = DEFAULT_STIMULUS_LABEL  # "fam+"
KEEP_TAIL_SEC = 0.1  # continuous data kept immediately before each onset
PRE_WINDOW_SEC = None  # None -> shortest remaining lead-in across the group
POST_WINDOW_SEC = None  # None -> shortest remaining lead-out across the group
DATA_STAGE = PreprocessedDataVariants.RAW_AFTER_ICA

# ── ASSR response inspection ──────────────────────────────────────────────────
EPOCH_TMIN, EPOCH_TMAX = -0.1, 1.2  # window around each fam+ onset (s)
ASSR_FREQ = 40.0  # expected steady-state frequency (Hz)

# ── Plot saving ───────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "stimulus_alignment"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Group: {EXPERIMENT.value} / {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Plots -> {PLOTS_DIR}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
print(f"Total recordings parsed: {len(dataset_handler.dataset_metadata)}")
dataset_handler.dataset_metadata.head()

## Run Alignment

Loads the `after_ica` recordings for the selected group, trims/splices them, and
returns the per-group metadata, the aligned recordings, and the fitted
`StimulusAligner` (which holds the alignment plan).

In [ ]:
filtered_df, aligned, aligner = dataset_handler.align_stimuli_by_annotations(
    music_type=MUSIC_TYPE,
    condition_type=CONDITION,
    exclusion_categories=EXCLUSION_CATEGORIES,
    stimulus_label=STIMULUS_LABEL,
    keep_tail_sec=KEEP_TAIL_SEC,
    pre_window_sec=PRE_WINDOW_SEC,
    post_window_sec=POST_WINDOW_SEC,
    data_type_to_load=DATA_STAGE,
)

labels = filtered_df[SingleDataMetadata.PARTICIPANT_ID].astype(str).tolist()
sfreq = aligner.sfreq
common_count = aligner.common_count

print(f"Recordings in group   : {len(aligned)}")
print(f"Common stimulus count : {common_count} (originals: {aligner.original_counts})")
print(f"Aligned length        : {aligner.total_length / sfreq:.2f} s "
      f"({aligner.total_length} samples)")
print(f"Pre / post kept        : {aligner.pre_target / sfreq:.3f} s / "
      f"{aligner.post_target / sfreq:.3f} s")

## Per-Recording Summary

In [ ]:
summary = pd.DataFrame(
    {
        "participant": labels,
        "condition": filtered_df[SingleDataMetadata.CONDITION]
        .map(lambda c: c.value)
        .tolist(),
        "orig_fam_count": aligner.original_counts,
        "loaded_length_s": [n / sfreq for n in aligner.recording_lengths],
        "aligned_length_s": [a.n_times / sfreq for a in aligned],
        "removed_s": [
            (n - aligner.total_length) / sfreq for n in aligner.recording_lengths
        ],
    }
)
print(f"All aligned lengths equal: {len({a.n_times for a in aligned}) == 1}")
summary

## Inter-Stimulus Intervals

Original inter-onset intervals per recording (truncated to the common count) and
the per-interval **target** (group minimum) the alignment trims everything down to.

In [ ]:
# Original onset gaps (seconds), per recording, over the common stimulus count.
orig_onsets = [o[:common_count] for o in aligner.onset_samples]
gaps = [np.diff(o) / sfreq for o in orig_onsets]
targets = aligner.interval_targets / sfreq

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Left: gap per interval index, one line per recording, plus the target.
for label, g in zip(labels, gaps):
    axes[0].plot(g, alpha=0.4, lw=0.8)
axes[0].plot(targets, color="black", lw=2.0, label="target (group min)")
axes[0].set_title("Inter-stimulus interval per index")
axes[0].set_xlabel("Stimulus interval index")
axes[0].set_ylabel("Interval (s)")
axes[0].legend()

# Right: distribution of all original gaps vs targets.
axes[1].hist(
    np.concatenate(gaps), bins=60, alpha=0.6, label="original gaps (all recordings)"
)
axes[1].hist(targets, bins=60, alpha=0.6, label="targets (per-interval min)")
axes[1].set_title("Interval distribution")
axes[1].set_xlabel("Interval (s)")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "inter_stimulus_intervals.png", dpi=150)
plt.show()

## Onset Alignment (before vs after)

Onset time relative to the first onset, per recording. Before alignment the
recordings drift apart as intervals accumulate; after alignment every recording
shares the same onset positions (all lines collapse onto one).

In [ ]:
idx = np.arange(common_count)
before = [(o - o[0]) / sfreq for o in orig_onsets]  # per recording
after = (aligner.aligned_onset_samples - aligner.aligned_onset_samples[0]) / sfreq

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for label, b in zip(labels, before):
    axes[0].plot(idx, b, alpha=0.5, lw=0.8)
axes[0].set_title("Before: onset time (rel. to first)")
axes[0].set_xlabel("Stimulus index")
axes[0].set_ylabel("Time since first onset (s)")

for label in labels:  # all identical -> draw once per recording to show overlap
    axes[1].plot(idx, after, alpha=0.5, lw=0.8)
axes[1].set_title("After: onset time (rel. to first) — all aligned")
axes[1].set_xlabel("Stimulus index")

plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "onset_alignment_before_after.png", dpi=150)
plt.show()

# Cumulative drift across recordings (spread at the last stimulus).
last_spread_before = max(b[-1] for b in before) - min(b[-1] for b in before)
print(f"Spread at last stimulus — before: {last_spread_before:.3f} s, after: 0.000 s")

## Verify Alignment

In [ ]:
# Compare the first `common_count` onsets: a recording may keep a trailing extra
# onset (beyond the common count) within its post-window, which is expected.
aligned_onsets = [
    get_stimulus_onset_samples(a, STIMULUS_LABEL)[:common_count] for a in aligned
]
identical = all(
    np.array_equal(aligned_onsets[0], other) for other in aligned_onsets[1:]
)
extra = [len(get_stimulus_onset_samples(a, STIMULUS_LABEL)) - common_count for a in aligned]
print(f"First {common_count} onset positions identical across recordings: {identical}")
print(f"Matches the planned positions: "
      f"{np.array_equal(aligned_onsets[0], aligner.aligned_onset_samples)}")
print(f"Trailing extra onsets kept per recording (beyond common count): {extra}")
print(f"First / last aligned onset (samples): "
      f"{aligner.aligned_onset_samples[0]} / {aligner.aligned_onset_samples[-1]}")

## ASSR Response

Epoch the aligned recordings around each `fam+` onset and average. The auditory
steady-state response should show power concentrated near the stimulation frequency
(~40 Hz). Seam annotations from splicing are ignored (`reject_by_annotation=False`)
since artifacts were already handled in preprocessing.

In [ ]:
# Epoch each recording around its fam+ onsets, then grand-average the per-recording
# evokeds (robust to per-subject info differences, and the standard cross-subject
# approach). `fam+` is regex-escaped for events_from_annotations.
label_regexp = rf"^{STIMULUS_LABEL.replace('+', chr(92) + '+')}$"

epochs_list = []
evokeds = []
for raw in aligned:
    events, _ = mne.events_from_annotations(raw, regexp=label_regexp)
    if len(events) == 0:
        continue
    ep = mne.Epochs(
        raw,
        events,
        tmin=EPOCH_TMIN,
        tmax=EPOCH_TMAX,
        baseline=(EPOCH_TMIN, 0.0),
        picks="eeg",
        reject_by_annotation=False,
        preload=True,
    )
    epochs_list.append(ep)
    evokeds.append(ep.average())

print(f"Recordings epoched: {len(evokeds)} | epochs each: {[len(e) for e in epochs_list]}")
grand = mne.grand_average(evokeds)

fig = grand.plot(spatial_colors=True, show=False)
fig.suptitle("Grand-average ASSR evoked response")
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_evoked.png", dpi=150)
plt.show()

### Response Spectrum

In [ ]:
# Per-recording epoch-mean spectrum (over epochs and channels), then averaged
# across recordings; expect a peak near ASSR_FREQ.
freqs = None
spectra = []
for ep in epochs_list:
    psd = ep.compute_psd(fmin=2.0, fmax=80.0)
    psds, freqs = psd.get_data(return_freqs=True)
    spectra.append(psds.mean(axis=(0, 1)))
mean_psd = np.mean(spectra, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freqs, 10 * np.log10(mean_psd))
ax.axvline(ASSR_FREQ, color="red", ls="--", lw=1, label=f"{ASSR_FREQ:.0f} Hz")
ax.set_title("Epoch-average power spectrum (across channels)")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power (dB)")
ax.legend()
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_spectrum.png", dpi=150)
plt.show()

### Response Time-Frequency

Time-frequency representation of the evoked (phase-locked) response. Morlet
wavelets over 2–80 Hz, computed per recording then grand-averaged (consistent
with the evoked grand-average above) and baseline-corrected (log-ratio) against
the pre-stimulus window. The ASSR should appear as a sustained band of power at
`ASSR_FREQ` (~40 Hz) for the duration of the stimulus.

In [ ]:
# Per-recording evoked TFR (Morlet), grand-averaged across recordings. n_cycles
# scales with frequency for a sensible time/frequency trade-off; the lower bound
# and cycle count are kept small enough that the wavelets fit the short epoch.
tfr_freqs = np.arange(4.0, 81.0, 1.0)
n_cycles = tfr_freqs / 4.0

evoked_tfrs = [
    ev.compute_tfr(
        method="morlet",
        freqs=tfr_freqs,
        n_cycles=n_cycles,
        verbose=False,
    )
    for ev in evokeds
]
grand_tfr = mne.grand_average(evoked_tfrs)
grand_tfr.apply_baseline(baseline=(EPOCH_TMIN, 0.0), mode="logratio")

# Average across channels into a single time-frequency map.
fig = grand_tfr.plot(
    combine="mean",
    title="Grand-average evoked TFR (across channels)",
    show=False,
)[0]
fig.axes[0].axhline(ASSR_FREQ, color="red", ls="--", lw=1)
# plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_evoked_tfr.png", dpi=150)
plt.show()